In [6]:
%pip install mlflow lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import sys
!{sys.executable} -m pip install mlflow lightgbm


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import mlflow
import lightgbm
print("MLflow version:", mlflow.__version__)
print("LightGBM version:", lightgbm.__version__)

MLflow version: 3.16.1
LightGBM version: 4.6.0


In [10]:
import os
import sys
import json
import hashlib
import platform
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import mlflow
from mlflow.tracking import MlflowClient

# 1. Configurazione MLflow locale (senza server esterno)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

SEED = 42

def load_data(data_path: str):
    """Carica il dataset e prepara il target binario."""
    df = pd.read_csv(data_path)
    if "class" in df.columns and "target" not in df.columns:
        df["target"] = (df["class"].str.strip().isin([">50K", ">50K."])).astype(int)
    return df

def split_data(df: pd.DataFrame, seed: int = SEED):
    """Divisione basata sulla colonna split fornita o su person_id."""
    if "split" in df.columns:
        train_mask = df["split"] == "train"
        val_mask = df["split"] == "val"
        test_mask = df["split"] == "test"
    else:
        # Fallback se split non è presente esplicitamente
        from sklearn.model_selection import GroupShuffleSplit
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        train_idx, test_idx = next(gss.split(df, groups=df["person_id"]))
        train_mask = df.index.isin(train_idx)
        val_mask = df.index.isin(test_idx[:len(test_idx)//2])
        test_mask = df.index.isin(test_idx[len(test_idx)//2:])

    drop_cols = [c for c in ["target", "class", "split", "person_id"] if c in df.columns]
    features = [c for c in df.columns if c not in drop_cols]

    return {
        "train": (df.loc[train_mask, features], df.loc[train_mask, "target"]),
        "val": (df.loc[val_mask, features], df.loc[val_mask, "target"]),
        "test": (df.loc[test_mask, features], df.loc[test_mask, "target"]),
        "features": features,
        "full_df": df
    }

def build_pipeline(params: dict, cat_cols=None, num_cols=None):
    """Costruisce la pipeline completa Scikit-Learn con LightGBM."""
    if cat_cols is None:
        cat_cols = []
    if num_cols is None:
        num_cols = []
        
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    
    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ], remainder="drop")
    
    clf = lgb.LGBMClassifier(**params)
    
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", clf)
    ])

print("Setup e funzioni base completate.")

Setup e funzioni base completate.


In [18]:
import os
import sys
import json
import hashlib
import platform
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import mlflow

SEED = 42

# --- 1. Veri Dosyasını Bul ---
csv_candidates = list(Path(r"C:\Users\nsupa").rglob("adult_income_issues.csv"))
if not csv_candidates:
    raise FileNotFoundError("adult_income_issues.csv bulunamadı!")
data_path = csv_candidates[0]
print(f"Veri yolu: {data_path}")

df = pd.read_csv(data_path)

# Target düzenleme
if "class" in df.columns and "target" not in df.columns:
    df["target"] = (df["class"].astype(str).str.strip().isin([">50K", ">50K."])).astype(int)

# --- 2. Split Kontrolü ve Güvenli Bölme ---
if "split" in df.columns:
    df["split_clean"] = df["split"].astype(str).str.strip().str.lower()
    print("Mevcut split değerleri ve sayıları:")
    print(df["split_clean"].value_counts())
    
    train_mask = df["split_clean"] == "train"
    val_mask = df["split_clean"].isin(["val", "validation", "valid"])
    test_mask = df["split_clean"] == "test"
else:
    train_mask = pd.Series(False, index=df.index)
    val_mask = pd.Series(False, index=df.index)
    test_mask = pd.Series(False, index=df.index)

# Eğer split kolonu yoksa veya val boşsa GroupShuffleSplit ile güvenli böl
if val_mask.sum() == 0:
    print("Uyarı: 'val' split'i bulunamadı, person_id üzerinden gruplu bölme yapılıyor...")
    from sklearn.model_selection import GroupShuffleSplit
    group_col = df["person_id"] if "person_id" in df.columns else df.index
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
    train_idx, temp_idx = next(gss.split(df, groups=group_col))
    
    # Kalan %30'u val ve test olarak ikiye böl
    val_idx = temp_idx[:len(temp_idx)//2]
    test_idx = temp_idx[len(temp_idx)//2:]
    
    train_mask = df.index.isin(train_idx)
    val_mask = df.index.isin(val_idx)
    test_mask = df.index.isin(test_idx)

drop_cols = [c for c in ["target", "class", "split", "split_clean", "person_id"] if c in df.columns]
feature_cols = [c for c in df.columns if c not in drop_cols]

X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, "target"]
X_val, y_val = df.loc[val_mask, feature_cols], df.loc[val_mask, "target"]
X_test, y_test = df.loc[test_mask, feature_cols], df.loc[test_mask, "target"]

print(f"Örneklem Sayıları -> Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

# --- 3. Pipeline Fabrikası ---
def build_pipeline(params):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])
    clf = lgb.LGBMClassifier(**params)
    return Pipeline([("preprocessor", preprocessor), ("classifier", clf)])

# --- 4. Part 0: Reproducibility Baseline ---
print("\n--- Part 0 Çalıştırılıyor ---")
det_params = {
    "n_estimators": 50,
    "random_state": SEED,
    "n_jobs": 1,
    "force_row_wise": True,
    "deterministic": True,
    "verbose": -1
}

pipe1 = build_pipeline(det_params).fit(X_train, y_train)
p1 = pipe1.predict_proba(X_val)[:, 1]
auc1, f1_1 = roc_auc_score(y_val, p1), f1_score(y_val, (p1 >= 0.5).astype(int))

pipe2 = build_pipeline(det_params).fit(X_train, y_train)
p2 = pipe2.predict_proba(X_val)[:, 1]
auc2, f1_2 = roc_auc_score(y_val, p2), f1_score(y_val, (p2 >= 0.5).astype(int))

print(f"Run 1 -> AUC: {auc1:.8f} | F1: {f1_1:.8f}")
print(f"Run 2 -> AUC: {auc2:.8f} | F1: {f1_2:.8f}")
is_exact = (auc1 == auc2) and (f1_1 == f1_2)
print(f"Birebir Aynı mı (Bit-for-bit Identical): {is_exact}")

# Hash hesaplama
hasher = hashlib.sha256()
with open(data_path, "rb") as f:
    while chunk := f.read(8192):
        hasher.update(chunk)
data_hash = hasher.hexdigest()

manifest = {
    "SEED": SEED,
    "data_file_sha256": data_hash,
    "python_version": platform.python_version(),
    "val_auc": float(auc1),
    "val_f1": float(f1_1),
    "deterministic_verified": bool(is_exact)
}

with open("reproducibility_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nreproducibility_manifest.json kaydedildi:")
print(json.dumps(manifest, indent=2))

Veri yolu: C:\Users\nsupa\Desktop\ml_industry_course\day1\generated\adult_income_issues.csv
Mevcut split değerleri ve sayıları:
split_clean
train    7397
test     1874
Name: count, dtype: int64
Uyarı: 'val' split'i bulunamadı, person_id üzerinden gruplu bölme yapılıyor...
Örneklem Sayıları -> Train: 6487, Val: 1392, Test: 1392

--- Part 0 Çalıştırılıyor ---


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Run 1 -> AUC: 1.00000000 | F1: 1.00000000
Run 2 -> AUC: 1.00000000 | F1: 1.00000000
Birebir Aynı mı (Bit-for-bit Identical): True

reproducibility_manifest.json kaydedildi:
{
  "SEED": 42,
  "data_file_sha256": "2be61444c6afc3bb2a3ca8a5f63ed643eb008d182c3d54ea083f50af2fc62288",
  "python_version": "3.14.3",
  "val_auc": 0.9999999999999999,
  "val_f1": 1.0,
  "deterministic_verified": true
}


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [20]:
# Hangi değişkenin sızıntı yaptığını gör
ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = list(ohe.get_feature_names_out(cat_cols))
all_features = num_cols + cat_feature_names

importances = pd.Series(
    best_pipeline.named_steps["classifier"].feature_importances_, 
    index=all_features
).sort_values(ascending=False)

print("En yüksek öneme sahip ilk 5 değişken:")
print(importances.head(5))

En yüksek öneme sahip ilk 5 değişken:
post_adjudication_risk_code    101
education_num                   55
db_row_surrogate_key            20
capital_gain                    18
capital_loss                     4
dtype: int32


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fit


--- Sızıntısız En İyi 5 Model (Validation) ---
        run_name  learning_rate  num_leaves  max_depth   val_auc    val_f1  \
4    clean_lgb_5            0.1          15         -1  0.913761  0.733491   
12  clean_lgb_13            0.1          15         -1  0.912777  0.734982   
6    clean_lgb_7            0.1          31         -1  0.912139  0.729191   
13  clean_lgb_14            0.1          15          5  0.912016  0.736718   
15  clean_lgb_16            0.1          31          5  0.911970  0.739694   

    val_auc_se  
4     0.009670  
12    0.009720  
6     0.009753  
13    0.009759  
15    0.009762  

Temiz Şampiyon Model: clean_lgb_5 (AUC: 0.91376)
Temiz Referans Model: clean_lgb_3 (AUC: 0.90513)
AUC Farkı: 0.00864 | %95 Güven Aralığı: [0.00447, 0.01303]

Nihai Test AUC: 0.91823


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import itertools
from sklearn.metrics import brier_score_loss

print("--- Part 1 Başlatılıyor: Sweep & Evaluation ---")

# 1. MLflow Deneyi Ayarla
EXPERIMENT_NAME = "day4_assignment_sweep"
mlflow.set_experiment(EXPERIMENT_NAME)

# 2. İstatistiksel Karşılaştırma Fonksiyonları
def hanley_mcneil_se(auc, n_pos, n_neg):
    """Hanley & McNeil (1982) formülüyle AUC standart hatası."""
    q1 = auc / (2.0 - auc)
    q2 = (2.0 * auc**2) / (1.0 + auc)
    numerator = auc * (1.0 - auc) + (n_pos - 1.0) * (q1 - auc**2) + (n_neg - 1.0) * (q2 - auc**2)
    return np.sqrt(max(0.0, numerator / (n_pos * n_neg)))

def bootstrap_auc_difference(y_true, p_a, p_b, n_bootstraps=1000, seed=SEED):
    """İki model arasındaki AUC farkının %95 güven aralığı."""
    rng = np.random.RandomState(seed)
    diffs = []
    n = len(y_true)
    y_true_arr = np.array(y_true)
    for _ in range(n_bootstraps):
        idx = rng.randint(0, n, size=n)
        if len(np.unique(y_true_arr[idx])) < 2:
            continue
        auc_a = roc_auc_score(y_true_arr[idx], p_a[idx])
        auc_b = roc_auc_score(y_true_arr[idx], p_b[idx])
        diffs.append(auc_a - auc_b)
    diffs = np.array(diffs)
    return float(np.mean(diffs)), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

# 3. Parametre Izgarası (Hyperparameter Grid)
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.03, 0.1],
    "num_leaves": [15, 31],
    "max_depth": [-1, 5]
}

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Toplam {len(combinations)} konfigürasyon taranacak.")

results = []
models = {}
val_preds_dict = {}

n_pos = int((y_val == 1).sum())
n_neg = int((y_val == 0).sum())

# 4. Sweep Döngüsü ve MLflow Günlüğü
for i, params in enumerate(combinations):
    run_name = f"lgb_run_{i+1}"
    full_params = {
        **params,
        "random_state": SEED,
        "n_jobs": 1,
        "force_row_wise": True,
        "deterministic": True,
        "verbose": -1
    }
    
    with mlflow.start_run(run_name=run_name):
        pipe = build_pipeline(full_params)
        pipe.fit(X_train, y_train)
        
        # Validation tahminleri
        val_probs = pipe.predict_proba(X_val)[:, 1]
        val_preds = (val_probs >= 0.5).astype(int)
        
        # Metrikler
        auc = float(roc_auc_score(y_val, val_probs))
        f1 = float(f1_score(y_val, val_preds))
        brier = float(brier_score_loss(y_val, val_probs))
        auc_se = float(hanley_mcneil_se(auc, n_pos, n_neg))
        
        # MLflow Log
        mlflow.log_params(params)
        mlflow.log_metrics({
            "val_auc": auc,
            "val_f1": f1,
            "val_brier": brier,
            "val_auc_se": auc_se
        })
        
        results.append({
            "run_name": run_name,
            **params,
            "val_auc": auc,
            "val_f1": f1,
            "val_brier": brier,
            "val_auc_se": auc_se
        })
        models[run_name] = pipe
        val_preds_dict[run_name] = val_probs

sweep_df = pd.DataFrame(results).sort_values(by="val_auc", ascending=False)
print("\n--- En İyi 5 Model (Validation AUC'ye göre) ---")
print(sweep_df[["run_name", "learning_rate", "num_leaves", "max_depth", "val_auc", "val_f1", "val_auc_se"]].head())

# 5. Baseline ile Şampiyon Modelin Karşılaştırılması
best_run = sweep_df.iloc[0]["run_name"]
baseline_run = sweep_df.iloc[-1]["run_name"] # En düşük ya da ilk model

p_best = val_preds_dict[best_run]
p_base = val_preds_dict[baseline_run]

mean_diff, ci_low, ci_high = bootstrap_auc_difference(y_val.to_numpy(), p_best, p_base)

print(f"\nŞampiyon Model: {best_run} (AUC: {sweep_df.iloc[0]['val_auc']:.5f})")
print(f"Referans Model: {baseline_run} (AUC: {sweep_df.iloc[-1]['val_auc']:.5f})")
print(f"AUC Farkı (Şampiyon - Referans): {mean_diff:.5f}")
print(f"%95 Güven Aralığı (Bootstrap CI): [{ci_low:.5f}, {ci_high:.5f}]")

# Test Seti Nihai Doğrulama
best_pipeline = models[best_run]
test_probs = best_pipeline.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_probs)
test_f1 = f1_score(y_test, (test_probs >= 0.5).astype(int))

print(f"\n--- Şampiyon Model Test Seti Sonucu ---")
print(f"Test AUC: {test_auc:.5f} | Test F1: {test_f1:.5f}")

--- Part 1 Başlatılıyor: Sweep & Evaluation ---


2026/09/22 18:27:41 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/22 18:27:41 INFO mlflow.store.db.utils: Updating database tables
2026/09/22 18:27:44 INFO mlflow.tracking.fluent: Experiment with name 'day4_assignment_sweep' does not exist. Creating a new experiment.


Toplam 16 konfigürasyon taranacak.


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fit


--- En İyi 5 Model (Validation AUC'ye göre) ---
      run_name  learning_rate  num_leaves  max_depth  val_auc  val_f1  \
0    lgb_run_1           0.03          15         -1      1.0     1.0   
1    lgb_run_2           0.03          15          5      1.0     1.0   
2    lgb_run_3           0.03          31         -1      1.0     1.0   
3    lgb_run_4           0.03          31          5      1.0     1.0   
11  lgb_run_12           0.03          31          5      1.0     1.0   

    val_auc_se  
0          0.0  
1          0.0  
2          0.0  
3          0.0  
11         0.0  

Şampiyon Model: lgb_run_1 (AUC: 1.00000)
Referans Model: lgb_run_5 (AUC: 1.00000)
AUC Farkı (Şampiyon - Referans): 0.00000
%95 Güven Aralığı (Bootstrap CI): [0.00000, 0.00000]

--- Şampiyon Model Test Seti Sonucu ---
Test AUC: 1.00000 | Test F1: 1.00000


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# --- Sızıntı Yaratan Değişkenleri Çıkararak Temiz Model Eğitimi ---
leakage_cols = ["post_adjudication_risk_code", "db_row_surrogate_key"]

drop_cols = [c for c in ["target", "class", "split", "split_clean", "person_id"] + leakage_cols if c in df.columns]
clean_features = [c for c in df.columns if c not in drop_cols]

X_train_clean = df.loc[train_mask, clean_features]
X_val_clean = df.loc[val_mask, clean_features]
X_test_clean = df.loc[test_mask, clean_features]

num_cols_clean = X_train_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_clean = X_train_clean.select_dtypes(exclude=[np.number]).columns.tolist()

def build_clean_pipeline(params):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols_clean),
        ("cat", cat_pipe, cat_cols_clean)
    ])
    clf = lgb.LGBMClassifier(**params)
    return Pipeline([("preprocessor", preprocessor), ("classifier", clf)])

# Temiz model ile Sweep
clean_results = []
clean_models = {}
clean_val_preds = {}

for i, params in enumerate(combinations):
    run_name = f"clean_lgb_{i+1}"
    full_params = {
        **params,
        "random_state": SEED,
        "n_jobs": 1,
        "force_row_wise": True,
        "deterministic": True,
        "verbose": -1
    }
    
    with mlflow.start_run(run_name=run_name):
        pipe = build_clean_pipeline(full_params)
        pipe.fit(X_train_clean, y_train)
        
        val_probs = pipe.predict_proba(X_val_clean)[:, 1]
        val_preds = (val_probs >= 0.5).astype(int)
        
        auc = float(roc_auc_score(y_val, val_probs))
        f1 = float(f1_score(y_val, val_preds))
        brier = float(brier_score_loss(y_val, val_probs))
        auc_se = float(hanley_mcneil_se(auc, n_pos, n_neg))
        
        mlflow.log_params(params)
        mlflow.log_metrics({
            "val_auc": auc,
            "val_f1": f1,
            "val_brier": brier,
            "val_auc_se": auc_se
        })
        
        clean_results.append({
            "run_name": run_name,
            **params,
            "val_auc": auc,
            "val_f1": f1,
            "val_brier": brier,
            "val_auc_se": auc_se
        })
        clean_models[run_name] = pipe
        clean_val_preds[run_name] = val_probs

clean_df = pd.DataFrame(clean_results).sort_values(by="val_auc", ascending=False)
print("\n--- Sızıntısız En İyi 5 Model (Validation) ---")
print(clean_df[["run_name", "learning_rate", "num_leaves", "max_depth", "val_auc", "val_f1", "val_auc_se"]].head())

# Şampiyon ve Referans Karşılaştırması
best_clean_run = clean_df.iloc[0]["run_name"]
base_clean_run = clean_df.iloc[-1]["run_name"]

diff, ci_l, ci_u = bootstrap_auc_difference(
    y_val.to_numpy(), 
    clean_val_preds[best_clean_run], 
    clean_val_preds[base_clean_run]
)

print(f"\clean best Model: {best_clean_run} (AUC: {clean_df.iloc[0]['val_auc']:.5f})")
print(f"clean reference Model: {base_clean_run} (AUC: {clean_df.iloc[-1]['val_auc']:.5f})")
print(f"AUC difference: {diff:.5f} | %95 Güven Aralığı: [{ci_l:.5f}, {ci_u:.5f}]")

# Test Seti Skoru
test_p = clean_models[best_clean_run].predict_proba(X_test_clean)[:, 1]
print(f"\final Test AUC: {roc_auc_score(y_test, test_p):.5f}")

c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fit


--- Sızıntısız En İyi 5 Model (Validation) ---
        run_name  learning_rate  num_leaves  max_depth   val_auc    val_f1  \
4    clean_lgb_5            0.1          15         -1  0.913761  0.733491   
12  clean_lgb_13            0.1          15         -1  0.912777  0.734982   
6    clean_lgb_7            0.1          31         -1  0.912139  0.729191   
13  clean_lgb_14            0.1          15          5  0.912016  0.736718   
15  clean_lgb_16            0.1          31          5  0.911970  0.739694   

    val_auc_se  
4     0.009670  
12    0.009720  
6     0.009753  
13    0.009759  
15    0.009762  

Temiz Şampiyon Model: clean_lgb_5 (AUC: 0.91376)
Temiz Referans Model: clean_lgb_3 (AUC: 0.90513)
AUC Farkı: 0.00864 | %95 Güven Aralığı: [0.00447, 0.01303]

Nihai Test AUC: 0.91823


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [24]:
# --- Part 2: Model Registry, Final Evaluation & Report Generation ---

import json
from pathlib import Path
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, log_loss

print("--- Starting Part 2: Final Evaluation and Artifact Packaging ---")

# 1. Retrieve the champion model pipeline
champion_run_name = best_clean_run
champion_pipeline = clean_models[champion_run_name]

# 2. Compute final evaluation metrics on the Test set
test_probs = champion_pipeline.predict_proba(X_test_clean)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

n_test_pos = int((y_test == 1).sum())
n_test_neg = int((y_test == 0).sum())

test_auc = float(roc_auc_score(y_test, test_probs))
test_auc_se = float(hanley_mcneil_se(test_auc, n_test_pos, n_test_neg))
test_f1 = float(f1_score(y_test, test_preds))
test_acc = float(accuracy_score(y_test, test_preds))
test_prec = float(precision_score(y_test, test_preds))
test_rec = float(recall_score(y_test, test_preds))
test_loss = float(log_loss(y_test, test_probs))
test_brier = float(brier_score_loss(y_test, test_probs))

# 3. Log Champion Model as MLflow Production Candidate
with mlflow.start_run(run_name=f"{champion_run_name}_champion_final"):
    # Log hyperparameters and full evaluation metrics
    champion_params = clean_df.loc[clean_df["run_name"] == champion_run_name].to_dict(orient="records")[0]
    mlflow.log_params({
        k: v for k, v in champion_params.items() 
        if k not in ["run_name", "val_auc", "val_f1", "val_brier", "val_auc_se"]
    })
    
    mlflow.log_metrics({
        "test_auc": test_auc,
        "test_auc_se": test_auc_se,
        "test_f1": test_f1,
        "test_accuracy": test_acc,
        "test_precision": test_prec,
        "test_recall": test_rec,
        "test_log_loss": test_loss,
        "test_brier_score": test_brier,
        "val_test_auc_gap": float(test_auc - champion_params["val_auc"])
    })
    
    # Save model using cloudpickle to avoid skops untrusted types validation
    mlflow.sklearn.log_model(
        sk_model=champion_pipeline,
        name="champion_model",
        serialization_format="cloudpickle"
    )

# 4. Generate Final Assignment Summary Report
report = {
    "project_metadata": {
        "dataset_path": str(data_path),
        "dataset_sha256": data_hash,
        "seed": SEED,
        "dropped_leakage_features": leakage_cols
    },
    "champion_model_config": {
        "run_name": champion_run_name,
        "learning_rate": champion_params["learning_rate"],
        "num_leaves": champion_params["num_leaves"],
        "max_depth": champion_params["max_depth"],
        "n_estimators": champion_params["n_estimators"]
    },
    "validation_performance": {
        "val_auc": champion_params["val_auc"],
        "val_auc_se": champion_params["val_auc_se"],
        "val_f1": champion_params["val_f1"],
        "val_brier": champion_params["val_brier"]
    },
    "statistical_comparison_vs_baseline": {
        "baseline_run_name": base_clean_run,
        "auc_difference_mean": diff,
        "bootstrap_95_ci_lower": ci_l,
        "bootstrap_95_ci_upper": ci_u,
        "statistically_significant": bool(ci_l > 0.0)
    },
    "final_test_set_metrics": {
        "test_roc_auc": round(test_auc, 5),
        "test_auc_standard_error": round(test_auc_se, 5),
        "test_f1_score": round(test_f1, 5),
        "test_accuracy": round(test_acc, 5),
        "test_precision": round(test_prec, 5),
        "test_recall": round(test_rec, 5),
        "test_brier_score": round(test_brier, 5),
        "test_log_loss": round(test_loss, 5)
    }
}

report_path = Path("day4_final_report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"\nSuccessfully logged model artifacts and created '{report_path.name}'.")
print("\n--- Final Performance Summary ---")
print(f"Model ID                  : {champion_run_name}")
print(f"Validation AUC            : {champion_params['val_auc']:.5f} (SE: {champion_params['val_auc_se']:.5f})")
print(f"Test AUC                  : {test_auc:.5f} (SE: {test_auc_se:.5f})")
print(f"Test F1-Score             : {test_f1:.5f}")
print(f"Test Brier Score          : {test_brier:.5f}")
print(f"Bootstrap AUC Delta (95%): +{diff:.5f} [{ci_l:.5f}, {ci_u:.5f}]")
print(f"Statistically Significant : {ci_l > 0.0}")

--- Starting Part 2: Final Evaluation and Artifact Packaging ---


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/09/22 18:41:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Successfully logged model artifacts and created 'day4_final_report.json'.

--- Final Performance Summary ---
Model ID                  : clean_lgb_5
Validation AUC            : 0.91376 (SE: 0.00967)
Test AUC                  : 0.91823 (SE: 0.00962)
Test F1-Score             : 0.73789
Test Brier Score          : 0.10514
Bootstrap AUC Delta (95%): +0.00864 [0.00447, 0.01303]
Statistically Significant : True


In [27]:
# --- Part 3: Threshold Calibration, Drift Simulation/Detection & Retrain Loop ---

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score, brier_score_loss
import json
from pathlib import Path
import mlflow

print("--- Starting Part 3: Production Governance Pipeline ---")

# ==============================================================================
# 1. THRESHOLD CALIBRATION
# ==============================================================================
print("\n[1/3] Calibrating Decision Threshold on Validation Set...")

val_probs = np.asarray(champion_pipeline.predict_proba(X_val_clean)[:, 1], dtype=np.float64)
test_probs = np.asarray(champion_pipeline.predict_proba(X_test_clean)[:, 1], dtype=np.float64)
y_val_arr = np.asarray(y_val, dtype=np.int32)
y_test_arr = np.asarray(y_test, dtype=np.int32)

precisions, recalls, thresholds = precision_recall_curve(y_val_arr, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

best_idx = int(np.argmax(f1_scores))
calibrated_threshold = float(thresholds[best_idx])
calibrated_val_f1 = float(f1_scores[best_idx])

default_test_preds = (test_probs >= 0.5).astype(int)
calibrated_test_preds = (test_probs >= calibrated_threshold).astype(int)

f1_default = float(f1_score(y_test_arr, default_test_preds))
f1_calibrated = float(f1_score(y_test_arr, calibrated_test_preds))

print(f"Optimal Threshold (Val F1 Max) : {calibrated_threshold:.4f}")
print(f"Validation F1 at Optimal Thresh : {calibrated_val_f1:.4f}")
print(f"Test F1 (Default 0.50 Threshold): {f1_default:.4f}")
print(f"Test F1 (Calibrated Threshold)  : {f1_calibrated:.4f}")

# ==============================================================================
# 2. DRIFT SIMULATOR & DETECTOR SUITE
# ==============================================================================
print("\n[2/3] Simulating Production Data Drift & Running Detection Suite...")

def calculate_psi(expected, actual, num_buckets=10):
    exp = np.asarray(expected[~np.isnan(expected)], dtype=np.float64)
    act = np.asarray(actual[~np.isnan(actual)], dtype=np.float64)
    if len(exp) == 0 or len(act) == 0:
        return 0.0
    eps = 1e-4
    quantiles = np.linspace(0, 100, num_buckets + 1)
    bins = np.percentile(exp, quantiles)
    bins[0] = -np.inf
    bins[-1] = np.inf
    
    exp_counts, _ = np.histogram(exp, bins=bins)
    act_counts, _ = np.histogram(act, bins=bins)
    
    exp_pct = exp_counts / len(exp) + eps
    act_pct = act_counts / len(act) + eps
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

X_prod_clean = X_test_clean.copy()
X_prod_drifted = X_test_clean.copy()

# Güvenli dönüştürme: metinleri NaN yaparak kayma ekle
if "age" in X_prod_drifted.columns:
    numeric_age = pd.to_numeric(X_prod_drifted["age"], errors="coerce")
    X_prod_drifted["age"] = numeric_age + 15.0

if "capital_gain" in X_prod_drifted.columns:
    numeric_cg = pd.to_numeric(X_prod_drifted["capital_gain"], errors="coerce")
    X_prod_drifted["capital_gain"] = numeric_cg * 1.5

detection_results = {}
for feat in num_cols_clean:
    if feat in X_prod_clean.columns:
        ref_vals = pd.to_numeric(X_train_clean[feat], errors="coerce").dropna().to_numpy()
        curr_vals = pd.to_numeric(X_prod_drifted[feat], errors="coerce").dropna().to_numpy()
        
        ks_res = ks_2samp(ref_vals, curr_vals)
        psi = calculate_psi(ref_vals, curr_vals)
        detection_results[feat] = {
            "ks_stat": float(ks_res.statistic),
            "p_value": float(ks_res.pvalue),
            "psi": float(psi),
            "drift_detected": bool(ks_res.pvalue < 0.05 or psi > 0.25)
        }

prod_clean_probs = np.asarray(champion_pipeline.predict_proba(X_prod_clean)[:, 1], dtype=np.float64)
prod_drifted_probs = np.asarray(champion_pipeline.predict_proba(X_prod_drifted)[:, 1], dtype=np.float64)

pred_psi = calculate_psi(prod_clean_probs, prod_drifted_probs)
pred_ks = ks_2samp(prod_clean_probs, prod_drifted_probs)
pred_pval = float(pred_ks.pvalue)

print(f"Prediction Output PSI     : {pred_psi:.4f} (Threshold > 0.25 indicates significant drift)")
print(f"Prediction Output KS p-val: {pred_pval:.4e}")
drift_triggered = bool(pred_psi > 0.25 or pred_pval < 0.05)
print(f"Automated Drift Alert     : {'TRIGGERED' if drift_triggered else 'NORMAL'}")

# ==============================================================================
# 3. AUTOMATED RETRAIN LOOP TRIGGER
# ==============================================================================
print("\n[3/3] Evaluating Automated Retraining Loop...")

retrain_summary = {}

if drift_triggered:
    print("Drift threshold exceeded! Triggering retrain loop on updated window...")
    
    X_retrain = pd.concat([X_train_clean.reset_index(drop=True), X_prod_drifted.reset_index(drop=True)], ignore_index=True)
    y_retrain = pd.concat([y_train.reset_index(drop=True), y_test.reset_index(drop=True)], ignore_index=True)
    
    retrained_pipeline = build_clean_pipeline(champion_params)
    retrained_pipeline.fit(X_retrain, y_retrain)
    
    retrained_val_probs = np.asarray(retrained_pipeline.predict_proba(X_val_clean)[:, 1], dtype=np.float64)
    retrained_val_auc = float(roc_auc_score(y_val_arr, retrained_val_probs))
    
    with mlflow.start_run(run_name="retrained_model_v2"):
        mlflow.log_params(champion_params)
        mlflow.log_metrics({
            "retrained_val_auc": retrained_val_auc,
            "prediction_psi_trigger": pred_psi
        })
        mlflow.sklearn.log_model(
            sk_model=retrained_pipeline,
            name="retrained_model_v2",
            serialization_format="cloudpickle"
        )
    
    retrain_summary = {
        "retrain_triggered": True,
        "trigger_cause": "Covariate & Prediction Output Drift",
        "retrained_dataset_rows": int(len(X_retrain)),
        "retrained_val_auc": round(retrained_val_auc, 5),
        "status": "New Champion Candidate Logged to MLflow"
    }
    print(f"Retrained Model Val AUC: {retrained_val_auc:.5f} -> Successfully Logged to MLflow")
else:
    retrain_summary = {
        "retrain_triggered": False,
        "status": "System stable, no retraining required"
    }
    print("No retraining required. Current model remains in production.")

# ==============================================================================
# 4. EXPORT GOVERNANCE METADATA TO JSON
# ==============================================================================
governance_report = {
    "threshold_calibration": {
        "calibrated_threshold": round(calibrated_threshold, 4),
        "f1_at_default_0_50": round(f1_default, 4),
        "f1_at_calibrated": round(f1_calibrated, 4),
        "metric_delta": round(f1_calibrated - f1_default, 4)
    },
    "drift_detection": {
        "prediction_psi": round(pred_psi, 4),
        "prediction_ks_p_value": float(pred_pval),
        "drift_alert": drift_triggered,
        "feature_metrics": detection_results
    },
    "retrain_loop": retrain_summary
}

with open("day4_governance_report.json", "w") as f:
    json.dump(governance_report, f, indent=2)

print(f"\nAll operations completed. Saved governance summary to 'day4_governance_report.json'.")

--- Starting Part 3: Production Governance Pipeline ---

[1/3] Calibrating Decision Threshold on Validation Set...


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Optimal Threshold (Val F1 Max) : 0.2739
Validation F1 at Optimal Thresh : 0.7761
Test F1 (Default 0.50 Threshold): 0.7379
Test F1 (Calibrated Threshold)  : 0.7480

[2/3] Simulating Production Data Drift & Running Detection Suite...


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Prediction Output PSI     : 0.0912 (Threshold > 0.25 indicates significant drift)
Prediction Output KS p-val: 8.5577e-01
Automated Drift Alert     : NORMAL

[3/3] Evaluating Automated Retraining Loop...
No retraining required. Current model remains in production.

All operations completed. Saved governance summary to 'day4_governance_report.json'.


c:\Users\nsupa\ml_industry_course\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
